# Interactive t-SNE coverage tuner

Manually tune the t-SNE projection (perplexity / seed / colouring / point size) for the
FTRL state-coverage plots and pick what looks best.

It reads the **cached features** written next to each figure (`{env}.npz` now stores the
pre-t-SNE `features` plus `algo` and `rounds`), so you can re-fit t-SNE at any perplexity
**without** re-running experiments or needing the raw scratch demos.

**Setup:** point `COVERAGE_DIR` (next cell) at a pulled coverage-plots directory, e.g.
`experiments/learning_curves/classical/plots/coverage`. If `{env}.npz` predates this
feature, regenerate it once with `plot_tsne_coverage` (it now caches `features`).

In [ ]:
%matplotlib inline
import pathlib

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import ipywidgets as widgets
from IPython.display import display

from imitation.experiments.ftrl import tsne_coverage
from imitation.experiments.ftrl.plot_tsne_coverage import (
    _auto_point_size,
    _unique_positions,
)

# EDIT ME: directory holding the cached {env}.npz (features + algo + rounds).
COVERAGE_DIR = pathlib.Path("experiments/learning_curves/classical/plots/coverage")

In [ ]:
def available_envs(cov_dir=COVERAGE_DIR):
    return sorted(p.stem for p in pathlib.Path(cov_dir).glob("*.npz"))


def load_env(env, cov_dir=COVERAGE_DIR):
    d = np.load(pathlib.Path(cov_dir) / f"{env}.npz", allow_pickle=True)
    if "features" not in d:
        raise KeyError(
            f"{env}.npz has no 'features' array. Regenerate it with the updated "
            "plot_tsne_coverage (it now caches the pre-t-SNE features)."
        )
    return d["features"], d["algo"].astype(str), d["rounds"].astype(int)


print("Envs found:", available_envs())

In [ ]:
def fit_and_render(env, perplexity=30, seed=0, point_size=0, color_by="arrival round"):
    feats, algo, rounds = load_env(env)
    n = len(feats)
    perp = min(perplexity, max(5.0, (n - 1) / 3.0))
    emb = TSNE(
        n_components=2, perplexity=perp, init="pca", random_state=seed, n_iter=1000
    ).fit_transform(feats)
    uniq_m = tsne_coverage.coverage_metrics_unique(feats, algo)
    knn_m = tsne_coverage.coverage_metrics_highdim(feats, algo)
    algos = sorted(set(algo.tolist()))
    ncols = min(3, len(algos))
    nrows = int(np.ceil(len(algos) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows), squeeze=False)
    xlim = (emb[:, 0].min(), emb[:, 0].max())
    ylim = (emb[:, 1].min(), emb[:, 1].max())
    xr = max(xlim[1] - xlim[0], 1e-9)
    yr = max(ylim[1] - ylim[0], 1e-9)
    jr = np.random.default_rng(0)
    n_rounds = max(int(rounds.max()), 1)
    sc = None
    for i, a in enumerate(algos):
        ax = axes[i // ncols][i % ncols]
        m = algo == a
        npts = int(m.sum())
        xs = emb[m, 0].astype(float)
        ys = emb[m, 1].astype(float)
        nu = _unique_positions(np.column_stack([xs, ys]))
        s = point_size if point_size and point_size > 0 else _auto_point_size(nu)
        if npts > 2 * max(nu, 1):  # jitter piled-up points so they are visible
            xs = xs + jr.normal(0, 0.008 * xr, npts)
            ys = ys + jr.normal(0, 0.008 * yr, npts)
        if color_by == "algo":
            sc = ax.scatter(xs, ys, s=s, alpha=0.6, edgecolors="none", color=f"C{i}")
        else:
            sc = ax.scatter(
                xs, ys, c=rounds[m], cmap="coolwarm", vmin=0, vmax=n_rounds,
                s=s, alpha=0.6, edgecolors="none",
            )
        ax.set_title(f"{a}  (n={npts}, uniq={uniq_m[a]}, kNN={knn_m[a]:.2f})")
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xticks([])
        ax.set_yticks([])
    for j in range(len(algos), nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")
    if color_by != "algo" and sc is not None:
        cb = fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.6)
        cb.set_label("Data Arrival Rounds")
    fig.suptitle(f"{env}  (perplexity={perp:.0f}, seed={seed}, colour={color_by})")
    return fig

In [ ]:
env_w = widgets.Dropdown(options=available_envs(), description="env")
perp_w = widgets.IntSlider(value=30, min=5, max=50, step=1, description="perplexity")
seed_w = widgets.IntSlider(value=0, min=0, max=9, step=1, description="seed")
color_w = widgets.Dropdown(options=["arrival round", "algo"], description="colour by")
psize_w = widgets.FloatText(value=0, description="pt size (0=auto)")
go = widgets.Button(description="Recompute t-SNE", button_style="primary")
save = widgets.Button(description="Save PNG")
out = widgets.Output()
_last = {}


def _run(_=None):
    with out:
        out.clear_output(wait=True)
        fig = fit_and_render(
            env_w.value, perp_w.value, seed_w.value, psize_w.value, color_w.value
        )
        _last["fig"], _last["env"] = fig, env_w.value
        plt.show()


def _save(_=None):
    with out:
        if "fig" in _last:
            p = pathlib.Path(f"{_last['env']}_tuned_p{perp_w.value}_s{seed_w.value}.png")
            _last["fig"].savefig(p, dpi=150)
            print("saved", p.resolve())


go.on_click(_run)
save.on_click(_save)
display(
    widgets.HBox([env_w, color_w, psize_w]),
    widgets.HBox([perp_w, seed_w]),
    widgets.HBox([go, save]),
    out,
)
_run()

### Notes
- **Click *Recompute t-SNE*** after changing perplexity/seed (t-SNE is re-fit; ~seconds
  for a few thousand points). Colour/point-size also re-render on recompute.
- **uniq** in each title = distinct visited states (the coverage-breadth signal);
  **kNN** = mean 5-NN distance in feature space. Interactive methods (ftl/ftrl) have
  higher `uniq`/`kNN` than the expert-distribution baselines (bc/bc_dagger).
- **Colour = algo** is useful when arrival-round colouring isn't informative (the
  interactive methods spread states across all rounds).
- **Save PNG** writes the current figure to the working directory.